# Model Testing & Evaluation
Test semua model (ViT, Swin, DeiT) pada test dataset dan visualisasi hasil prediksi

## Import Required Libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import glob
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Setup matplotlib
import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries imported successfully!")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

✅ Libraries imported successfully!
GPU Available: True
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   GPU Memory: 4.29 GB


## Setup Configuration & Load Test Dataset

In [2]:
# Import model functions
from model import get_model, count_parameters

# Define class names
CLASS_NAMES = ['bakso', 'gado_gado', 'nasi_goreng', 'rendang', 'soto_ayam']
NUM_CLASSES = 5

# Setup paths
RESULTS_DIR = 'results'
TEST_DIR = 'test'
MODELS_CONFIG = {
    'ViT': {'folder': 'ViT', 'model_name': 'vit'},
    'Swin': {'folder': 'Swin', 'model_name': 'swin'},
    'DeiT': {'folder': 'DeiT', 'model_name': 'deit'}
}

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"✅ Configuration loaded!")
print(f"   Classes: {CLASS_NAMES}")
print(f"   Models: {list(MODELS_CONFIG.keys())}")
print(f"   Test directory: {TEST_DIR}")
print(f"   Device: {device}")

✅ Configuration loaded!
   Classes: ['bakso', 'gado_gado', 'nasi_goreng', 'rendang', 'soto_ayam']
   Models: ['ViT', 'Swin', 'DeiT']
   Test directory: test
   Device: cuda


In [3]:
class TestDataset(Dataset):
    """Load test images without labels"""
    def __init__(self, test_dir, transform=None):
        self.transform = transform
        self.image_files = sorted(glob.glob(os.path.join(test_dir, '*.jpg')))
        self.image_names = [os.path.basename(f) for f in self.image_files]
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        img = Image.open(self.image_files[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.image_names[idx]

# Define transform
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

# Load test dataset
test_dataset = TestDataset(TEST_DIR, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=0)

print(f"✅ Test dataset loaded!")
print(f"   Total images: {len(test_dataset)}")
print(f"   Batch size: 8")
print(f"   Total batches: {len(test_loader)}")

✅ Test dataset loaded!
   Total images: 10
   Batch size: 8
   Total batches: 2


## Define Model Loading & Inference Functions

In [4]:
def find_best_model_path(model_folder):
    """Find best model checkpoint in weights folder"""
    weights_dir = os.path.join(RESULTS_DIR, model_folder, 'weights')
    model_files = glob.glob(os.path.join(weights_dir, 'best_*.pth'))
    
    if not model_files:
        print(f"❌ No model found in {weights_dir}")
        return None
    
    model_path = model_files[0]
    return model_path

def load_model(model_name, model_path, device):
    """Load model dengan weight dari checkpoint"""
    model = get_model(model_name.lower(), num_classes=NUM_CLASSES)
    
    if not os.path.exists(model_path):
        print(f"❌ Model file not found: {model_path}")
        return None
    
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint)
    model = model.to(device)
    model.eval()
    
    return model

def run_inference(model, test_loader, device):
    """Run inference on test data"""
    predictions = []
    confidences = []
    image_names = []
    
    with torch.no_grad():
        for images, names in test_loader:
            images = images.to(device)
            
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            
            conf, preds = torch.max(probs, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            confidences.extend(conf.cpu().numpy())
            image_names.extend(names)
    
    return predictions, confidences, image_names

print("✅ Functions defined successfully!")

✅ Functions defined successfully!


## Load All Models & Run Predictions

In [5]:
# Load all models and run predictions
all_results = {}

for model_short_name, config in MODELS_CONFIG.items():
    print(f"\n{'='*60}")
    print(f"Processing {model_short_name}")
    print(f"{'='*60}")
    
    # Find best model path
    model_path = find_best_model_path(config['folder'])
    if model_path is None:
        continue
    
    print(f"✓ Model: {model_path}")
    
    # Load model
    model = load_model(config['model_name'], model_path, device)
    if model is None:
        continue
    
    # Run inference
    predictions, confidences, image_names = run_inference(model, test_loader, device)
    
    # Store results
    all_results[model_short_name] = {
        'predictions': predictions,
        'confidences': confidences,
        'image_names': image_names,
        'model_path': model_path
    }
    
    print(f"✓ Predictions completed")
    print(f"  Total predictions: {len(predictions)}")

print(f"\n{'='*60}")
print(f"✓ All models processed successfully!")
print(f"{'='*60}")


Processing ViT
✓ Model: results\ViT\weights\best_vit_epoch10.pth
✓ Predictions completed
  Total predictions: 10

Processing Swin
✓ Model: results\Swin\weights\best_swin_epoch3.pth
✓ Predictions completed
  Total predictions: 10

Processing DeiT
✓ Model: results\DeiT\weights\best_deit_epoch5.pth
✓ Predictions completed
  Total predictions: 10

✓ All models processed successfully!


## Save Results to CSV Files

In [6]:
def save_results_to_csv(results_dict):
    """Save predictions to CSV files"""
    for model_name, results in results_dict.items():
        predictions = results['predictions']
        confidences = results['confidences']
        image_names = results['image_names']
        
        # Create DataFrame
        df = pd.DataFrame({
            'image_name': image_names,
            'predicted_class': [CLASS_NAMES[p] for p in predictions],
            'predicted_idx': predictions,
            'confidence': confidences
        })
        
        # Save to CSV
        csv_filename = f'jawaban_test{model_name}.csv'
        df.to_csv(csv_filename, index=False)
        
        print(f"✓ Saved {model_name} results to {csv_filename}")

# Save results
print("Saving results to CSV files...")
print("="*60)
save_results_to_csv(all_results)
print("="*60)
print("✓ All results saved!")

Saving results to CSV files...
✓ Saved ViT results to jawaban_testViT.csv
✓ Saved Swin results to jawaban_testSwin.csv
✓ Saved DeiT results to jawaban_testDeiT.csv
✓ All results saved!


## Detailed Prediction Results for Selected Images

In [7]:
# Create a detailed comparison for selected images
selected_images = ['0003.jpg', '0006.jpg', '0012.jpg', '0017.jpg', '0022.jpg']

# Create a mapping from image names to predictions
image_predictions = {}
for model_name, results in all_results.items():
    for img_name, pred_idx, conf in zip(results['image_names'], 
                                         results['predictions'], 
                                         results['confidences']):
        if img_name not in image_predictions:
            image_predictions[img_name] = {}
        image_predictions[img_name][model_name] = {
            'class': CLASS_NAMES[pred_idx],
            'confidence': conf
        }

print("\n" + "="*80)
print("DETAILED PREDICTIONS FOR SELECTED IMAGES")
print("="*80)

for img_name in selected_images:
    if img_name in image_predictions:
        print(f"\n📸 {img_name}")
        print("-" * 80)
        for model_name in MODELS_CONFIG.keys():
            if model_name in image_predictions[img_name]:
                pred_info = image_predictions[img_name][model_name]
                class_name = pred_info['class']
                conf = pred_info['confidence']
                print(f"   {model_name:<8} → {class_name:<15} (Confidence: {conf:.4f} / {conf*100:.2f}%)")


DETAILED PREDICTIONS FOR SELECTED IMAGES

📸 0003.jpg
--------------------------------------------------------------------------------
   ViT      → soto_ayam       (Confidence: 0.9695 / 96.95%)
   Swin     → gado_gado       (Confidence: 0.9991 / 99.91%)
   DeiT     → gado_gado       (Confidence: 0.9975 / 99.75%)

📸 0006.jpg
--------------------------------------------------------------------------------
   ViT      → nasi_goreng     (Confidence: 0.6343 / 63.43%)
   Swin     → nasi_goreng     (Confidence: 0.9999 / 99.99%)
   DeiT     → nasi_goreng     (Confidence: 0.9965 / 99.65%)

📸 0012.jpg
--------------------------------------------------------------------------------
   ViT      → bakso           (Confidence: 0.7149 / 71.49%)
   Swin     → bakso           (Confidence: 0.9996 / 99.96%)
   DeiT     → bakso           (Confidence: 0.9533 / 95.33%)

📸 0017.jpg
--------------------------------------------------------------------------------
   ViT      → soto_ayam       (Confidence: 0.7

## Visualize Selected Images with All Model Predictions

In [8]:
fig, axes = plt.subplots(len(selected_images), 2, figsize=(14, 15))

for idx, img_name in enumerate(selected_images):
    if img_name not in image_predictions:
        continue
    
    # Load and display image
    img_path = os.path.join(TEST_DIR, img_name)
    img = Image.open(img_path)
    
    axes[idx, 0].imshow(img)
    axes[idx, 0].set_title(f"{img_name}", fontsize=12, fontweight='bold')
    axes[idx, 0].axis('off')
    
    # Create prediction table
    ax_table = axes[idx, 1]
    ax_table.axis('off')
    
    # Prepare table data
    table_data = []
    table_data.append(['Model', 'Prediction', 'Confidence'])
    
    for model_name in MODELS_CONFIG.keys():
        if model_name in image_predictions[img_name]:
            pred_info = image_predictions[img_name][model_name]
            class_name = pred_info['class']
            conf = pred_info['confidence']
            table_data.append([model_name, class_name, f"{conf:.2%}"])
    
    # Create table
    table = ax_table.table(cellText=table_data, cellLoc='center', loc='center',
                          colWidths=[0.25, 0.4, 0.35])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.5)
    
    # Style header row
    for i in range(3):
        table[(0, i)].set_facecolor('#4CAF50')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    # Alternate row colors
    for i in range(1, len(table_data)):
        for j in range(3):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#f0f0f0')
            else:
                table[(i, j)].set_facecolor('#ffffff')

plt.tight_layout()
plt.savefig('selected_predictions_detail.png', dpi=100, bbox_inches='tight')
print("✓ Saved detailed predictions visualization to selected_predictions_detail.png")
plt.show()

✓ Saved detailed predictions visualization to selected_predictions_detail.png


## Summary Statistics & Analysis

In [9]:
print("\n" + "="*80)
print("PREDICTION SUMMARY")
print("="*80)

summary_data = []

for model_name, results in all_results.items():
    predictions = results['predictions']
    confidences = results['confidences']
    
    summary_data.append({
        'Model': model_name,
        'Total Predictions': len(predictions),
        'Avg Confidence': np.mean(confidences),
        'Min Confidence': np.min(confidences),
        'Max Confidence': np.max(confidences),
        'Std Dev': np.std(confidences)
    })

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

# Show class distribution
print("\n" + "="*80)
print("CLASS DISTRIBUTION PER MODEL")
print("="*80)

for model_name, results in all_results.items():
    predictions = results['predictions']
    unique, counts = np.unique(predictions, return_counts=True)
    
    print(f"\n{model_name}:")
    for class_idx, count in zip(unique, counts):
        class_name = CLASS_NAMES[class_idx]
        percentage = (count / len(predictions)) * 100
        print(f"  {class_name:<15} : {count:3d} ({percentage:5.1f}%)")

print("\n" + "="*80)


PREDICTION SUMMARY
Model  Total Predictions  Avg Confidence  Min Confidence  Max Confidence  Std Dev
  ViT                 10        0.827863        0.634279        0.999252 0.126410
 Swin                 10        0.992684        0.937545        0.999996 0.018467
 DeiT                 10        0.977444        0.833607        0.999595 0.049813

CLASS DISTRIBUTION PER MODEL

ViT:
  bakso           :   3 ( 30.0%)
  gado_gado       :   1 ( 10.0%)
  nasi_goreng     :   1 ( 10.0%)
  rendang         :   2 ( 20.0%)
  soto_ayam       :   3 ( 30.0%)

Swin:
  bakso           :   2 ( 20.0%)
  gado_gado       :   2 ( 20.0%)
  nasi_goreng     :   2 ( 20.0%)
  rendang         :   2 ( 20.0%)
  soto_ayam       :   2 ( 20.0%)

DeiT:
  bakso           :   2 ( 20.0%)
  gado_gado       :   2 ( 20.0%)
  nasi_goreng     :   2 ( 20.0%)
  rendang         :   2 ( 20.0%)
  soto_ayam       :   2 ( 20.0%)



## Confidence Distribution Visualization

In [10]:
fig, axes = plt.subplots(1, len(all_results), figsize=(5*len(all_results), 5))

if len(all_results) == 1:
    axes = [axes]

for idx, (model_name, results) in enumerate(all_results.items()):
    confidences = results['confidences']
    
    axes[idx].hist(confidences, bins=20, alpha=0.7, color='steelblue', edgecolor='black')
    axes[idx].set_xlabel('Confidence Score', fontsize=12)
    axes[idx].set_ylabel('Frequency', fontsize=12)
    axes[idx].set_title(f'{model_name}\nConfidence Distribution', fontsize=13, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    
    # Add statistics
    mean_conf = np.mean(confidences)
    median_conf = np.median(confidences)
    axes[idx].axvline(mean_conf, color='red', linestyle='--', label=f'Mean: {mean_conf:.3f}')
    axes[idx].axvline(median_conf, color='green', linestyle='--', label=f'Median: {median_conf:.3f}')
    axes[idx].legend()

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=100, bbox_inches='tight')
print("✓ Saved confidence distribution plot")
plt.show()

✓ Saved confidence distribution plot


## Class Distribution Visualization

In [11]:
fig, axes = plt.subplots(1, len(all_results), figsize=(6*len(all_results), 5))

if len(all_results) == 1:
    axes = [axes]

for idx, (model_name, results) in enumerate(all_results.items()):
    predictions = results['predictions']
    unique, counts = np.unique(predictions, return_counts=True)
    
    class_labels = [CLASS_NAMES[i] for i in unique]
    
    axes[idx].bar(class_labels, counts, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_ylabel('Count', fontsize=12)
    axes[idx].set_title(f'{model_name}\nPredicted Class Distribution', fontsize=13, fontweight='bold')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add count labels on bars
    for i, (label, count) in enumerate(zip(class_labels, counts)):
        axes[idx].text(i, count + 0.1, str(count), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('class_distribution.png', dpi=100, bbox_inches='tight')
print("✓ Saved class distribution plot")
plt.show()

✓ Saved class distribution plot
